# LangChain: LCEL & Chains & Parsers

## Outline
* LCEL philosophy — pipe operator
* Building a simple chain
* Chain with output parser
* Sequential chains
* Parallel chains with RunnableParallel
* Router chain with Pydantic


## Installation

In [20]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5.2", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3", model_provider="ollama", temperature=0)


## 1. LCEL Philosophy

LCEL (LangChain Expression Language)



In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# The simplest chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
])

# pipe operator
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"input": "What is the capital of Iran?"})
print(result)
print(f"\nType: {type(result)}")  # str — not AIMessage


The capital of Iran is **Tehran**.

Type: <class 'langchain_core.messages.base.TextAccessor'>


In [3]:
# streaming with chain
print("Streaming output:")
for chunk in chain.stream({"input": "Tell me a short jok"}):
    print(chunk, end="", flush=True)
print()


Streaming output:
Why don’t skeletons fight each other?  
They don’t have the guts.


In [4]:
# batch — multiple parallel requests
results = chain.batch([
    {"input": "Capital of France?"},
    {"input": "Capital of Germany?"},
    {"input": "Capital of Japan?"},
    {"input": "Capital of Iran?"},
])
for r in results:
    print(r)


Paris.
Berlin.
Tokyo.
Tehran.


## 2. Output Parsers

In [5]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

# JsonOutputParser — JSON output
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Return a JSON object with 'answer' (Yes/No) and 'confidence' (0-1) keys only. No markdown."),
    ("human", "{question}"),
])

json_chain = json_prompt | llm | JsonOutputParser()
result = json_chain.invoke({"question": "Is the earth round?"})
print(result)
print(f"Answer: {result['answer']}, Confidence: {result['confidence']}")


{'answer': 'Yes', 'confidence': 0.99}
Answer: Yes, Confidence: 0.99


In [21]:
json_chain = json_prompt | ollama | JsonOutputParser()
result = json_chain.invoke({"question": "Is the earth round?"})
print(result)
print(f"Answer: {result['answer']}, Confidence: {result['confidence']}")

{'answer': 'Yes', 'confidence': 1}
Answer: Yes, Confidence: 1


In [6]:
# Structured output with Pydantic — recommended method
class ReviewAnalysis(BaseModel):
    """Sentiment analysis of a review"""
    sentiment: str = Field(description="POSITIVE, NEGATIVE, or NEUTRAL")
    score: float = Field(description="Score from 0 to 1")
    key_points: list[str] = Field(description="Main points from the review")

structured_llm = llm.with_structured_output(ReviewAnalysis)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the sentiment of the given review."),
    ("human", "{review}"),
])

review_chain = review_prompt | structured_llm

result = review_chain.invoke({
    "review": "This product is amazing! Best purchase I've made this year. Super fast delivery too."
})
print(f"Sentiment: {result.sentiment}")
print(f"Score: {result.score}")
print(f"Key points: {result.key_points}")


Sentiment: POSITIVE
Score: 0.95
Key points: ['Product is described as amazing', 'Considered the best purchase made this year', 'Super fast delivery']


In [22]:
structured_llm = ollama.with_structured_output(ReviewAnalysis)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the sentiment of the given review."),
    ("human", "{review}"),
])

review_chain = review_prompt | structured_llm

result = review_chain.invoke({
    "review": "This product is amazing! Best purchase I've made this year. Super fast delivery too."
})
print(f"Sentiment: {result.sentiment}")
print(f"Score: {result.score}")
print(f"Key points: {result.key_points}")

Sentiment: POSITIVE
Score: 0.9
Key points: ['amazing', 'best purchase', 'super fast delivery']


## 3. Sequential Chains 

In [24]:
!ollama list

NAME             ID              SIZE      MODIFIED      
kimi-k3:cloud    e8aa77394b8b    -         2 weeks ago      
gpt-oss:120b     a951a23b46a1    65 GB     6 months ago     
llama3:latest    365c0bd3c000    4.7 GB    7 months ago     
gpt-oss:20b      aa4295ac10c3    13 GB     11 months ago    


In [23]:
# Sequential chain: the output of one chain becomes the input of the next
# Old: SimpleSequentialChain / SequentialChain
# New: multiple chains with | and RunnableLambda

from langchain_core.runnables import RunnableLambda

# Chain 1: Summary
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])
#summarize_chain = summarize_prompt | llm | StrOutputParser()
summarize_chain = summarize_prompt | ollama | StrOutputParser()

# Chain 2: Translate the summary
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to Persian. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
    ("human", "{text}"),
])
#translate_chain = translate_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | ollama | StrOutputParser()

# Combine the two chains
sequential_chain = (
    summarize_chain 
    | RunnableLambda(lambda x: {"text": x})  # output of the first chain → input of the second chain
    | translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy 
of the information the models generate. It includes APIs and integrations for a wide 
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("Persian summary:")
print(result)


Persian summary:
زنجیره لنگ یک فریمورک است که توسعه برنامه‌های با مدل‌های زبان بزرگ را امکان‌پذیر می‌کند، ابزارها و ابزاری را فراهم می‌کند تا سفارشی، دقت و مرتبط بودن اطلاعات تولید شده را بهبود بخشد.


In [25]:

from langchain_core.runnables import RunnableLambda

# Chain 1: Summary
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one sentence."),
    ("human", "{text}"),
])
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Chain 2: Translate the summary
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following text to {lang}. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
    ("human", "{text}"),
])
translate_chain = translate_prompt | llm | StrOutputParser()

# Combine the two chains
sequential_chain = (
    summarize_chain 
    | RunnableLambda(lambda x: {"text": x, "lang":"French"})  # output of the first chain → input of the second chain
    | translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy 
of the information the models generate. It includes APIs and integrations for a wide 
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("Persian summary:")
print(result)


Persian summary:
LangChain est un framework permettant de créer des applications propulsées par des grands modèles de langage, offrant des outils, des API et des intégrations (p. ex., des bases de données vectorielles, des chargeurs de documents, des analyseurs de sortie) afin d’améliorer la personnalisation, la précision et la pertinence.


## 4. Parallel Chains 

In [26]:
from langchain_core.runnables import RunnableParallel

# Parallel: multiple chains run simultaneously
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 pros of the given product in bullet points."),
    ("human", "{product}"),
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 cons of the given product in bullet points."),
    ("human", "{product}"),
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a one-line summary of the given product."),
    ("human", "{product}"),
])

# All three chains run in parallel
parallel_chain = RunnableParallel(
    pros=(pros_prompt | ollama | StrOutputParser()),
    cons=(cons_prompt | ollama | StrOutputParser()),
    summary=(summary_prompt | ollama | StrOutputParser()),
)

result = parallel_chain.invoke({"product": "iPhone 15 Pro"})
print("=== Summary ===")
print(result["summary"])
print("\n=== Pros ===")
print(result["pros"])
print("\n=== Cons ===")
print(result["cons"])


=== Summary ===
The iPhone 15 Pro is a high-end smartphone featuring a 6.1-inch Super Retina XDR display, A16 Bionic chip, up to 16GB of RAM, triple-camera setup with a telephoto lens, and advanced features like MagSafe wireless charging and a durable stainless steel frame.

=== Pros ===
I apologize, but since the iPhone 15 Pro has not been released yet, I can only provide information on the previous models. Here are three pros of the iPhone 14 Pro:

• **Exceptional Cameras**: The iPhone 14 Pro has a quad-camera setup with a wide-angle lens, telephoto lens, and ultra-wide lens, allowing for high-quality photos and videos. The camera system also features a new feature called "Photographic Styles" that allows users to customize the look of their photos.

• **Fast Performance**: The iPhone 14 Pro is powered by Apple's A16 Bionic chip, which provides fast performance and efficient battery life. The phone also supports 5G connectivity and has a fast charging feature that can charge the batt

In [27]:
import time

# ====== 3 questions whose answers are exactly one word ======

# Question 1: Yes/No
q1_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just name of a city."),
    ("human", "What is the Capital of Iran?")
])

# Question 2: A number
q2_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD (a number like 'two' or 'five')."),
    ("human", "How many legs does a dog have?")
])

# Question 3: A simple name
q3_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just the name."),
    ("human", "What color is an apple?")
])

chain1 = q1_prompt | ollama | StrOutputParser()
chain2 = q2_prompt | ollama | StrOutputParser()
chain3 = q3_prompt | ollama | StrOutputParser()

# ====== 1. Normal execution 3 times sequentially ======
start_seq = time.time()

r1 = chain1.invoke({})
r2 = chain2.invoke({})
r3 = chain3.invoke({})

end_seq = time.time()

print("=== Normal Mode (Sequential) ===")
print(f"Answers: '{r1}', '{r2}', '{r3}'")
print(f"Time: {end_seq - start_seq:.3f} seconds\n")

# ====== 2. Parallel execution ======
start_par = time.time()

parallel_chain = RunnableParallel(
    ans1=chain1,
    ans2=chain2,
    ans3=chain3,
)

results = parallel_chain.invoke({})
end_par = time.time()

print("=== Parallel Mode ===")
print(f"Answers: '{results['ans1']}', '{results['ans2']}', '{results['ans3']}'")
print(f"Time: {end_par - start_par:.3f} seconds")
print(f"\n⚡ Improvement ratio: {(end_seq - start_seq) / (end_par - start_par):.2f}x")

=== Normal Mode (Sequential) ===
Answers: 'Tehran', 'Four', 'Red'
Time: 0.728 seconds

=== Parallel Mode ===
Answers: 'Tehran', 'Four', 'Red'
Time: 0.227 seconds

⚡ Improvement ratio: 3.21x


## 5. Router Chain — Replacement for RouterChain


<img src="./images/router.png" width="2000" height="600">

In [32]:
from pydantic import BaseModel
from typing import Literal

# Router: routes to the appropriate chain based on the topic
class RouteQuery(BaseModel):
    """Detect the topic of the question"""
    topic: Literal["math", "history", "science", "other"]

router_llm = llm.with_structured_output(RouteQuery)

# Specialized chains
math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math expert. Answer math questions step by step."),
    ("human", "{question}"),
])

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a history expert. Provide historical context."),
    ("human", "{question}"),
])

science_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science expert. Explain scientific concepts clearly."),
    ("human", "{question}"),
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])

# Router Function
def route(input_dict):      
    """Detect the topic and select the appropriate chain"""
    route_result = router_llm.invoke(input_dict["question"])
    topic = route_result.topic
    
    chains = {
        "math": math_prompt | ollama | StrOutputParser(),
        "history": history_prompt | ollama | StrOutputParser(),
        "science": science_prompt | ollama | StrOutputParser(),
        "other": general_prompt | ollama | StrOutputParser(),
    }
    
    print(f"→ Routed to: {topic}")
    return chains[topic].invoke(input_dict["question"])

# Test the router
questions = [
    "What is the derivative of x^2?",
    "When did World War II end?",
    "What is the best programming language?",
    "How does photosynthesis work?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {route({'question': q})[:400]}...")



Q: What is the derivative of x^2?
→ Routed to: math
A: To find the derivative of x^2, we can use the power rule of differentiation, which states that if we have a function of the form f(x) = x^n, then the derivative is f'(x) = nx^(n-1).

In this case, we have f(x) = x^2, so we can apply the power rule as follows:

f(x) = x^2
f'(x) = 2x^(2-1)
= 2x^1
= 2x

So, the derivative of x^2 is 2x....

Q: When did World War II end?
→ Routed to: history
A: World War II ended on September 2, 1945, when Japan formally surrendered to the Allied Powers on board the USS Missouri, a United States Navy battleship, in Tokyo Bay, Japan. This marked the end of the war in the Pacific and the final victory of the Allies over the Axis powers.

However, the war in Europe had ended earlier, on May 8, 1945, with the formal surrender of Germany, known as V-E Day (Vi...

Q: What is the best programming language?
→ Routed to: other
A: The answer, of course, is that there is no single "best" programming language. The